In [1]:
import  pandas as pd
import numpy   as np
import os
import glob
import sqlite3
data_path = './data/'
csv_files = glob.glob(data_path + '*.csv')
dataframe={}
for file in csv_files:
    name= os.path.basename(file).replace(".csv","")
    dataframe[name] = pd.read_csv(file)

In [2]:
#提取商品表和订单商品明细表
products = dataframe["olist_products_dataset"]
items = dataframe["olist_order_items_dataset"]

In [3]:
#查看商品表
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [4]:
#商品订单表
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [5]:
products_data = pd.merge(
    items,
    products,
    on = "product_id",
    how = "left"
)
print("合并前商品明细行数",len(items))
print("合并后商品行数",len(products_data))
products_data.head()
products.shape


合并前商品明细行数 112650
合并后商品行数 112650


(32951, 9)

In [6]:
#未匹配的商品信息
print("未匹配的商品信息",
      products_data["product_category_name"].isna().sum())

未匹配的商品信息 1603


In [7]:
#检查商品编号是否匹配
products_data = pd.merge(
    items,
    products,
    on = "product_id",
    how = "left",
    indicator = True
)
print(products_data["_merge"].value_counts())#
#根据结果，发现商品全匹配，但是空值的存在可能数据原本就是空值



_merge
both          112650
left_only          0
right_only         0
Name: count, dtype: int64


In [8]:
#确定商品缺失数量
print("商品类别缺失数量：",
      products_data["product_category_name"].isna().sum())
print("商品重量缺失数量：",
      products_data["product_weight_g"].isna().sum())
print("product_name_lenght ",products_data["product_name_lenght"].isna().sum())
print("product_description_lenght ",products_data["product_description_lenght"].isna().sum())


商品类别缺失数量： 1603
商品重量缺失数量： 18
product_name_lenght  1603
product_description_lenght  1603


In [9]:
print(products_data.isna().sum())

order_id                         0
order_item_id                    0
product_id                       0
seller_id                        0
shipping_limit_date              0
price                            0
freight_value                    0
product_category_name         1603
product_name_lenght           1603
product_description_lenght    1603
product_photos_qty            1603
product_weight_g                18
product_length_cm               18
product_height_cm               18
product_width_cm                18
_merge                           0
dtype: int64


In [10]:
missing_weight = products_data[
    products_data["product_weight_g"].isna()
]

print("重量缺失记录数：", len(missing_weight))
print("其中类别也缺失的数量：",
      missing_weight["product_category_name"].isna().sum())

重量缺失记录数： 18
其中类别也缺失的数量： 17


In [11]:
print(
    "至少缺失一个字段",
    products_data.isna().any(axis=1).sum()
)
#比1603多一条,而在重量中，存在18条缺失，且17条类别缺失，也就是说存在独一条重量缺失

至少缺失一个字段 1604


In [12]:
print(missing_weight.isna().sum())
#18个订单中，存在一个独缺长宽高和重量

order_id                       0
order_item_id                  0
product_id                     0
seller_id                      0
shipping_limit_date            0
price                          0
freight_value                  0
product_category_name         17
product_name_lenght           17
product_description_lenght    17
product_photos_qty            17
product_weight_g              18
product_length_cm             18
product_height_cm             18
product_width_cm              18
_merge                         0
dtype: int64


In [13]:
#统计销量最高的商品
product_sales = products_data.groupby(
    "product_id"
)["order_item_id"].count()
product_sales.head()

product_id
00066f42aeeb9f3007548bb9d3f33c38    1
00088930e925c41fd95ebfe695fd2655    1
0009406fd7479715e4bef61dd91f2462    1
000b8f95fcb9e0096488278317764d19    2
000d9be29b5207b54e86aa1b1ac54872    1
Name: order_item_id, dtype: int64

In [14]:
#最高销量的10个商品
top_products = product_sales.sort_values(
    ascending=False
)
top_products.head(10)
#只知道商品id不知道类别和信息

product_id
aca2eb7d00ea1a7b8ebd4e68314663af    527
99a4788cb24856965c36a24e339b6058    488
422879e10f46682990de24d770e7f83d    484
389d119b48cf3043d311335e499d9c6b    392
368c6c730842d78016ad823897a372db    388
53759a2ecddad2bb87a079a1f1519f73    373
d1c427060a0f73f6b889a5c7c61f2ac4    343
53b36df67ebb7c41585e8d54d6772e08    323
154e7e31ebfa092203795c972e5804a6    281
3dd2a17168ec895c781a9191c1e95ad7    274
Name: order_item_id, dtype: int64

In [15]:
#最高销售额前十
product_revenue = products_data.groupby(
    "product_id"
)["price"].sum()
top_revenue = product_revenue.sort_values(
    ascending=False
)
top_revenue.head(10)

product_id
bb50f2e236e5eea0100680137654686c    63885.00
6cdd53843498f92890544667809f1595    54730.20
d6160fb7873f184099d9bc95e30376af    48899.34
d1c427060a0f73f6b889a5c7c61f2ac4    47214.51
99a4788cb24856965c36a24e339b6058    43025.56
3dd2a17168ec895c781a9191c1e95ad7    41082.60
25c38557cf793876c5abdd5931f922db    38907.32
5f504b3a1c75b73d6151be81eb05bdc9    37733.90
53b36df67ebb7c41585e8d54d6772e08    37683.42
aca2eb7d00ea1a7b8ebd4e68314663af    37608.90
Name: price, dtype: float64

In [16]:
#销售量高不代表销售额高

In [17]:
#销售额第一商品的
top_revenue_id = product_revenue.idxmax()
top_revenue_product = products_data[
    products_data["product_id"] == top_revenue_id
]
print(top_revenue_product[[
    "product_id",
    "product_category_name",
    "price"
]].head())
#id相同的五个商品

                            product_id product_category_name  price
853   bb50f2e236e5eea0100680137654686c          beleza_saude  330.0
1007  bb50f2e236e5eea0100680137654686c          beleza_saude  325.0
3254  bb50f2e236e5eea0100680137654686c          beleza_saude  325.0
4334  bb50f2e236e5eea0100680137654686c          beleza_saude  330.0
4335  bb50f2e236e5eea0100680137654686c          beleza_saude  330.0


In [18]:
#为什么他的销售额高呢？
print("商品销量",len(top_revenue_product))#统计数量
print("平均单价",top_revenue_product["price"].mean())
print("销售额",top_revenue_product["price"].sum())
#销售量包括单价都挺高

商品销量 195
平均单价 327.61538461538464
销售额 63885.0


In [19]:
#比较销售额冠军和销售量冠军
top_sales_id = product_sales.idxmax()#提取销售量冠军id
top_sales_product = products_data[
    products_data["product_id"] == top_sales_id
]
print("销售量",len(top_sales_product))
print("平均单价",top_sales_product["price"].mean())
print("销售额",top_sales_product["price"].sum())

销售量 527
平均单价 71.36413662239089
销售额 37608.899999999994


In [20]:
#结论，销售量大不代表销售额大。商品分析不能只看销售量

In [21]:
#分析销售额中哪些商品贡献大
category_sales = products_data.groupby(
    "product_category_name"
)["price"].sum()
#提取其中的价格根据商品类别进行分类
category_sales = category_sales.sort_values(
    ascending=False
)
#排序一下
category_sales.head(10)

product_category_name
beleza_saude              1258681.34
relogios_presentes        1205005.68
cama_mesa_banho           1036988.68
esporte_lazer              988048.97
informatica_acessorios     911954.32
moveis_decoracao           729762.49
cool_stuff                 635290.85
utilidades_domesticas      632248.66
automotivo                 592720.11
ferramentas_jardim         485256.46
Name: price, dtype: float64

In [22]:
#但是我们知道了价格的高低，但我们不知道商品的单价高低，是否因为销量大还是单价高而贡献大

In [23]:
#先统计销量
category_quantity = products_data.groupby(
    "product_category_name"
)["order_item_id"].count()

top_category_quantity = category_quantity.sort_values(
    ascending=False
)
top_category_quantity.head(10)

product_category_name
cama_mesa_banho           11115
beleza_saude               9670
esporte_lazer              8641
moveis_decoracao           8334
informatica_acessorios     7827
utilidades_domesticas      6964
relogios_presentes         5991
telefonia                  4545
ferramentas_jardim         4347
automotivo                 4235
Name: order_item_id, dtype: int64

In [24]:
#cama_mesa_banho销量第3，但销售额第1，relogios_presentes销量第7，但销售额第2；cama_mesa_banho销售额第3但销售量第1

In [25]:
#平均单价计算
category_avg_price = (products_data.groupby(
    "product_category_name"
)["price"].mean())
category_avg_price.sort_values(ascending=False).head(10)

product_category_name
pcs                                              1098.340542
portateis_casa_forno_e_cafe                       624.285658
eletrodomesticos_2                                476.124958
agro_industria_e_comercio                         342.124858
instrumentos_musicais                             281.616000
eletroportateis                                   280.778468
portateis_cozinha_e_preparadores_de_alimentos     264.568667
telefonia_fixa                                    225.693182
construcao_ferramentas_seguranca                  208.992371
relogios_presentes                                201.135984
Name: price, dtype: float64

In [26]:
#在三个表中，只找到relogios_presentes，也就是说cama_mesa_banho的销售优势可能是大规模的销售量

In [27]:
#结合三表
products_data
category_analysis = products_data.groupby(
    "product_category_name"
).agg(
    sales_quantity=("order_item_id", "count"),#统计该类型商品（根据分类的字段）订单数，也就是销量
    total_sales = ("price", "sum"),#统计该类型商品销售总额
    avg_price = ("price", "mean")#平均销售额
)
category_analysis = category_analysis.sort_values(
    "total_sales",
    ascending=False
)#按照总销售额进行排列
print(category_analysis.head(10))

                        sales_quantity  total_sales   avg_price
product_category_name                                          
beleza_saude                      9670   1258681.34  130.163531
relogios_presentes                5991   1205005.68  201.135984
cama_mesa_banho                  11115   1036988.68   93.296327
esporte_lazer                     8641    988048.97  114.344285
informatica_acessorios            7827    911954.32  116.513903
moveis_decoracao                  8334    729762.49   87.564494
cool_stuff                        3796    635290.85  167.357969
utilidades_domesticas             6964    632248.66   90.788148
automotivo                        4235    592720.11  139.957523
ferramentas_jardim                4347    485256.46  111.630196


In [28]:
#那以上就是代表综合表现
